# Discrimination pipeline

Run the built-in **discrimination** paradigm. Real StimPy sessions are resolved via `config.paths`; **if none are found (or a session fails to parse) we print the error and fall back to a simulated session** so every analysis cell still runs.

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs. (Real sessions need their full artifacts — e.g. opto-pattern images for opto sessions — or parsing will raise.)

In [ ]:
import os, glob
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.core.hub import Hub
from piepy.simulations.session import simulate_session
from piepy.stats import aggregate, Rate, Median
from piepy.fitting import fit

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Find sessions

In [ ]:
candidates = sorted(glob.glob(os.path.join(pres, "*discrim*")))
print(f"found {len(candidates)} candidate discrimination session(s)")
for c in candidates[:5]:
    print(' ', os.path.basename(c))

## Parse a single session
Builds the Session, parses each run, and stacks them onto one session clock. Wrapped defensively so a real-data hiccup prints a clear error instead of aborting.

In [ ]:

name = "250404_VB101_discrim_opto120_V1__no_cam_VO"
sess = get_session_class("discrimination")(name, load_flag=False)
df = sess.concatenate_runs("discrimination")
print(name, "->", df.shape)


## Cohort across many sessions (`Hub`)
`Hub` runs each session in parallel and stacks them into one cohort table. It already isolates per-session failures (a bad session warns and is skipped).

In [ ]:
cohort = None
if candidates:
    try:
        hub = Hub("discrimination")
        hub.initialize([os.path.basename(c) for c in candidates], load_sessions=False)
        cohort = hub.data
        print('cohort:', None if cohort is None else cohort.shape)
    except Exception as e:
        print("Hub gather failed:", type(e).__name__, e)

## Psychometric: P(rightward) vs signed contrast
`right_choice` is already 0/1, so the `rate=` shorthand works directly.

In [ ]:
data = df if df is not None else simulate_session(
    paradigm="discrimination", n_trials=900, seed=0)

# P(rightward choice) vs signed contrast
psy = aggregate(data, group=["signed_contrast"], rate="right_choice")
x, y = psy["signed_contrast"].to_numpy(), psy["value"].to_numpy()
n = psy["n"].to_numpy()
res = fit("logistic", x, y, n=n)

xx = np.linspace(x.min(), x.max(), 200)
plt.errorbar(x, y, yerr=[y - psy['ci_low'].to_numpy(), psy['ci_high'].to_numpy() - y],
             fmt="o", capsize=3, label="data")
plt.plot(xx, res.predict(xx), "-", label="logistic fit")
plt.xlabel("signed contrast"); plt.ylabel("P(rightward)")
plt.title("Discrimination psychometric"); plt.legend(); plt.show()

## Reaction time vs signed contrast
Same `aggregate` API, a `Median` metric with its order-statistic CI.

In [ ]:
rt = aggregate(data.filter(pl.col("outcome") == "correct"), group=["signed_contrast"], metrics=[Median("response_time")])
rt